# MaskFormer Training on Welding Robot Data

End-to-end training pipeline for `GLCAnomalyMaskFormer` using the mock welding robot dataset.

**Pipeline:**
1. Load and explore CSV (1.84M timesteps, 3 sensors, 250ms intervals)
2. Group consecutive timesteps into macro-windows (seq_len=240), with padding
3. Build `RawWindowDataset` with robot/program indices
4. Initialize MaskFormer (num_sensors=3)
5. Train with `MaskFormerTrainerRunner`
6. Visualize training curves

In [ ]:
# %% Imports
import sys
import os
import json
import math
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import matplotlib.pyplot as plt

# Add project root to path
project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from app.anomaly_detection.models import (
    create_model,
    filter_model_kwargs,
    MaskFormerModels,
)
from app.anomaly_detection.training.maskformer_trainer import (
    MaskFormerTrainerRunner,
)
from app.anomaly_detection.training.datasets import RawWindowDataset

print("All imports OK")
print(f"PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")


## 1. Load and Explore Data

In [ ]:
# Load the full CSV
CSV_PATH = project_root / "data" / "mock_welding_robot_data.csv"
print(f"Reading {CSV_PATH}...")

import csv
with open(CSV_PATH, "r") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(f"Total rows: {len(rows):,}")
print(f"Columns: {list(rows[0].keys())}")

# Sensor columns
SENSOR_COLS = ["wire_feed_speed_mm_min", "welding_voltage_v", "welding_current_a"]
print(f"Sensor columns: {SENSOR_COLS}")

# Quick stats
robot_codes = sorted(set(r["robot_code"] for r in rows))
program_nums = sorted(set(r["program_number"] for r in rows))
phases = sorted(set(r["phase"] for r in rows))
print(f"Robot codes: {robot_codes}")
print(f"Program numbers: {program_nums}")
print(f"Phases: {phases}")
print(f"Example row: {dict(list(rows[0].items())[:6])}")


## 2. Merge into Macro-Windows with Padding

Each "file" = unique `(robot_code, start_time, program_number)` group.
Rows within a group are ordered by `serial_number`.
We extract the 3 sensor values, zero-pad to `seq_len=240` if shorter, or slice if longer.

In [ ]:
SEQ_LEN = 240  # MaskFormer default macro-window length
NUM_SENSORS = 3  # wire_feed_speed, welding_voltage, welding_current

# Group rows by (robot_code, start_time, program_number)
groups = defaultdict(list)
for r in rows:
    key = (r["robot_code"], r["start_time"], r["program_number"])
    groups[key].append(r)

print(f"Total groups (files): {len(groups)}")

# Build windows: each group → one file-window [C, T]
windows_list = []
robot_idx_list = []
program_idx_list = []
group_keys = []

# Build robot and program codebooks
all_robot_codes = sorted(set(r["robot_code"] for r in rows))
all_program_nums = sorted(set(r["program_number"] for r in rows))
robot_to_idx = {code: i for i, code in enumerate(all_robot_codes)}
program_to_idx = {pn: i for i, pn in enumerate(all_program_nums)}

print(f"Robot codebook: {len(robot_to_idx)} codes → indices 0..{len(robot_to_idx)-1}")
print(f"Program codebook: {len(program_to_idx)} programs → indices 0..{len(program_to_idx)-1}")

for key, group_rows in groups.items():
    # Sort by serial_number (ascending)
    group_rows.sort(key=lambda x: int(x["serial_number"]))
    
    # Extract sensor values as float array [T, C]
    file_arr = np.array(
        [[float(r[col]) for col in SENSOR_COLS] for r in group_rows],
        dtype=np.float32,
    )  # [T, C]
    
    T = file_arr.shape[0]
    
    if T >= SEQ_LEN:
        # Take first SEQ_LEN steps
        window = file_arr[:SEQ_LEN, :].T  # [C, T]
    else:
        # Zero-pad to SEQ_LEN
        pad = np.zeros((SEQ_LEN - T, NUM_SENSORS), dtype=np.float32)
        window = np.vstack([file_arr, pad]).T  # [C, T]
    
    windows_list.append(window)
    robot_idx_list.append(robot_to_idx[key[0]])
    program_idx_list.append(program_to_idx[key[1]])
    group_keys.append(key)

# Stack into arrays
windows_arr = np.stack(windows_list, axis=0)  # [N, C, T]
robot_arr = np.array(robot_idx_list, dtype=np.int64)
program_arr = np.array(program_idx_list, dtype=np.int64)

print(f"\nWindows shape: {windows_arr.shape}  [N, C={NUM_SENSORS}, T={SEQ_LEN}]")
print(f"Robot indices shape: {robot_arr.shape}, unique: {np.unique(robot_arr)}")
print(f"Program indices shape: {program_arr.shape}, unique: {np.unique(program_arr)}")
print(f"Sensor value range: [{windows_arr.min():.4f}, {windows_arr.max():.4f}]")


In [ ]:
# Quick sanity: plot a few file-windows
fig, axes = plt.subplots(3, 3, figsize=(14, 8))
for i in range(3):
    for j in range(3):
        idx = i * 3 + j
        ax = axes[i, j]
        w = windows_arr[idx]
        for s in range(NUM_SENSORS):
            ax.plot(w[s], label=SENSOR_COLS[s].replace("_", " "))
        ax.set_title(f"Robot={group_keys[idx][0]} Prog={group_keys[idx][1]}")
        ax.legend(fontsize=7)
plt.tight_layout()
plt.show()


## 3. Create Dataset and DataLoader

In [ ]:
# Split into train/val (80/20)
np.random.seed(42)
N = len(windows_arr)
indices = np.random.permutation(N)
split = int(0.8 * N)
train_idx = indices[:split]
val_idx = indices[split:]

train_dataset = RawWindowDataset(
    windows=windows_arr[train_idx],
    robot_indices=robot_arr[train_idx],
    program_indices=program_arr[train_idx],
)
val_dataset = RawWindowDataset(
    windows=windows_arr[val_idx],
    robot_indices=robot_arr[val_idx],
    program_indices=program_arr[val_idx],
)

print(f"Train files: {len(train_dataset)}")
print(f"Val files:   {len(val_dataset)}")
print(f"Sample output keys: {list(train_dataset[0].keys())}")
print(f"  input shape: {train_dataset[0]['input'].shape}")
print(f"  robot_idx:   {train_dataset[0]['robot_idx']}")
print(f"  program_idx: {train_dataset[0]['program_idx']}")


## 4. Initialize MaskFormer Model

Key config adjustments for welding data:
- `num_sensors=3` (wire_feed_speed, welding_voltage, welding_current)
- `num_robots` set to actual unique robot count
- `num_programs` set to actual unique program count

In [ ]:
num_unique_robots = len(np.unique(robot_arr))
num_unique_programs = len(np.unique(program_arr))

config = {
    "num_sensors": NUM_SENSORS,
    "seq_len": SEQ_LEN,
    "num_robots": num_unique_robots,
    "num_programs": num_unique_programs,
    "use_context_embeddings": True,
    "d_model": 128,
    "transformer_layers": 4,
    "transformer_heads": 4,
    "mask_ratio": 0.3,
    "lambda_assoc": 2.0,
    "lambda_orth": 0.05,
}

model = create_model(MaskFormerModels.GLC_ANOMALY_MASKFORMER, config)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {model.name}")
print(f"Config: num_sensors={config['num_sensors']}, seq_len={config['seq_len']}, "
      f"d_model={config['d_model']}, num_robots={config['num_robots']}, "
      f"num_programs={config['num_programs']}")
print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")


## 5. Initialize Trainer and Train

In [ ]:
BATCH_SIZE = 32
EPOCHS = 30
LR = 1e-4

# Create DataLoaders
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
)

print(f"Batches per epoch: train={len(train_loader)}, val={len(val_loader)}")

# Build runner and train
runner = MaskFormerTrainerRunner(model_or_config=model)
history = runner.train(
    train_loader=list(train_loader),
    val_loader=list(val_loader),
    epochs=EPOCHS,
    lr=LR,
)

print("\nTraining complete!")
print(f"Final loss: {history['loss'][-1]:.4f}")
print(f"Final recon_loss: {history['recon_loss'][-1]:.4f}")
print(f"Final assoc_loss_min: {history['assoc_loss_min'][-1]:.4f}")
print(f"Final sigma_mean: {history['sigma_mean'][-1]:.3f}")


## 6. Visualize Training Curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
epochs = range(1, len(history["loss"]) + 1)

metrics = [
    ("loss", "Total Loss", None),
    ("recon_loss", "Reconstruction Loss", None),
    ("assoc_loss_min", "Association Loss (min)", None),
    ("assoc_loss_max", "Association Loss (max)", None),
    ("orth_loss", "Orthogonality Loss", None),
    ("sigma_mean", "Sigma Mean", "orange"),
]
for ax, (key, title, color) in zip(axes.flat, metrics):
    ax.plot(epochs, history[key], marker="o", color=color or "steelblue")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 7. Save Model Checkpoint

In [ ]:
# Save model checkpoint using BaseAnomalyDetectionModel.save_manual_checkpoint
ckpt_dir = project_root / "artifacts" / "maskformer_checkpoint"
model.save_manual_checkpoint(
    epoch=EPOCHS,
    val_loss=history["loss"][-1],
    folder_path=str(ckpt_dir),
    is_best=True,
)
print(f"Model state_dict saved to {ckpt_dir / f'model_{model.name}_best.pt'}")
print(f"Model metadata saved to {ckpt_dir / f'metadata_{model.name}_best.json'}")

# Also save training history and codebooks separately
extra = {
    "history": history,
    "robot_codebook": robot_to_idx,
    "program_codebook": program_to_idx,
}
with open(ckpt_dir / "training_extra.json", "w") as f:
    json.dump(extra, f, indent=2, default=str)
print(f"Training extras (history, codebooks) saved to {ckpt_dir / 'training_extra.json'}")
print("Done! 🎉")
